# Lesson 5a: Principal Component Analysis (PCA) — Theory

## Overview
Principal Component Analysis (PCA) is the foundational technique for linear dimensionality reduction. It finds the directions of maximum variance in data and projects data onto these directions. This lesson derives PCA from first principles using eigendecomposition and Singular Value Decomposition.

**Learning Goals:**
- Understand covariance matrices and variance maximization
- Derive principal components via eigendecomposition
- Connect SVD to PCA
- Compute explained variance and select number of components
- Implement PCA from scratch in NumPy
- Understand numerical stability issues

## Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import svd, eigh
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

## Part 1: Motivation — The Curse of Dimensionality

High-dimensional data is challenging:
- **Computational complexity**: storage, computation time
- **Statistical sparsity**: fewer samples relative to dimensions
- **Visualization**: impossible to visualize beyond 3 dimensions
- **Noise**: high dimensions increase noise influence

**Solution**: Reduce dimensions while preserving important information (variance).

In [ ]:
# Generate high-dimensional data with low intrinsic dimensionality
n_samples = 500
true_dims = 2
noise_dims = 8

# True signal: 2D manifold in 10D space
t1 = np.random.uniform(0, 2*np.pi, n_samples)
t2 = np.random.uniform(0, 2*np.pi, n_samples)

signal = np.column_stack([np.cos(t1), np.sin(t2)])
noise = np.random.randn(n_samples, noise_dims)
X = np.hstack([signal, noise])

print(f"Data shape: {X.shape}")
print(f"True intrinsic dimensionality: {true_dims}")
print(f"Ambient dimensionality: {X.shape[1]}")
print(f"True variance in signal: {signal.var():.3f}")
print(f"Average variance per noise dimension: {noise.var():.3f}")

# Visualize first 3 dimensions
fig = plt.figure(figsize=(12, 4))
ax1 = fig.add_subplot(131)
ax1.scatter(X[:, 0], X[:, 1], c=t1, cmap='hsv', alpha=0.6, s=30)
ax1.set_title('Dimensions 0-1 (Signal + Noise)')
ax1.set_xlabel('x₀')
ax1.set_ylabel('x₁')

ax2 = fig.add_subplot(132)
ax2.scatter(X[:, 0], X[:, 2], c=t1, cmap='hsv', alpha=0.6, s=30)
ax2.set_title('Dimensions 0-2 (Signal + Noise)')
ax2.set_xlabel('x₀')
ax2.set_ylabel('x₂')

ax3 = fig.add_subplot(133)
ax3.scatter(signal[:, 0], signal[:, 1], c=t1, cmap='hsv', alpha=0.6, s=30)
ax3.set_title('True 2D Manifold')
ax3.set_xlabel('True Dim 1')
ax3.set_ylabel('True Dim 2')

plt.tight_layout()
plt.show()

## Part 2: Covariance Matrix and Variance Maximization

### Centering the Data
First, center data by subtracting mean:
$$\mathbf{X}_{centered} = \mathbf{X} - \mathbf{1}\boldsymbol{\mu}^T$$

### Covariance Matrix
The sample covariance matrix is:
$$\mathbf{S} = \frac{1}{n-1}\mathbf{X}_{centered}^T\mathbf{X}_{centered} \in \mathbb{R}^{d \times d}$$

### Variance Maximization Objective
We seek a direction $\mathbf{w}$ that maximizes variance:
$$\max_{\mathbf{w}} \text{Var}(\mathbf{X}\mathbf{w}) = \max_{\mathbf{w}} \mathbf{w}^T\mathbf{S}\mathbf{w}$$
subject to $\|\mathbf{w}\| = 1$ (unit norm constraint).

**Solution**: $\mathbf{w}$ is the eigenvector of $\mathbf{S}$ with largest eigenvalue!

In [ ]:
# Create 2D synthetic data
np.random.seed(42)
n = 200
angle = np.pi / 6  # 30 degrees
cov = [[2, 1.5], [1.5, 1]]
X_2d = np.random.multivariate_normal([0, 0], cov, n)

# Compute covariance matrix
X_centered = X_2d - X_2d.mean(axis=0)
S = (X_centered.T @ X_centered) / (n - 1)

print("Sample Covariance Matrix:")
print(S)

# Eigendecomposition
eigenvalues, eigenvectors = np.linalg.eigh(S)
eigenvalues = eigenvalues[::-1]  # Sort descending
eigenvectors = eigenvectors[:, ::-1]

print(f"\nEigenvalues: {eigenvalues}")
print(f"\nEigenvectors (columns):")
print(eigenvectors)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Data and principal components
ax = axes[0]
ax.scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.5, s=30)

# Plot eigenvectors scaled by eigenvalues
for i in range(2):
    scale = 2 * np.sqrt(eigenvalues[i])
    ax.arrow(0, 0, scale * eigenvectors[0, i], scale * eigenvectors[1, i],
            head_width=0.2, head_length=0.1, fc=f'C{i}', ec=f'C{i}', linewidth=2,
            label=f'PC{i+1} (λ={eigenvalues[i]:.2f})')

ax.set_xlim(-3, 3)
ax.set_ylim(-2, 2)
ax.set_aspect('equal')
ax.set_xlabel('x₁')
ax.set_ylabel('x₂')
ax.set_title('Data with Principal Components')
ax.legend()
ax.grid(True)

# Variance explained
ax = axes[1]
explained_var = eigenvalues / eigenvalues.sum()
cumsum_var = np.cumsum(explained_var)

ax.bar(range(1, 3), explained_var, alpha=0.6, label='Individual')
ax.plot(range(1, 3), cumsum_var, 'ro-', linewidth=2, markersize=8, label='Cumulative')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance')
ax.set_title('Variance Explained')
ax.set_xticks([1, 2])
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 3: Eigendecomposition of Covariance Matrix

### Eigendecomposition
$$\mathbf{S} = \mathbf{U}\boldsymbol{\Lambda}\mathbf{U}^T$$

where:
- $\mathbf{U} = [\mathbf{u}_1, \mathbf{u}_2, \ldots, \mathbf{u}_d]$ are eigenvectors (principal components)
- $\boldsymbol{\Lambda} = \text{diag}(\lambda_1, \lambda_2, \ldots, \lambda_d)$ are eigenvalues (variances along components)
- Eigenvalues ordered: $\lambda_1 \geq \lambda_2 \geq \ldots \geq \lambda_d \geq 0$

### Projection onto Principal Components
Transform data to new coordinate system:
$$\mathbf{Z} = \mathbf{X}_{centered}\mathbf{U}_k$$

where $\mathbf{U}_k$ uses only the first $k$ eigenvectors (columns of $\mathbf{U}$).

### Explained Variance
Fraction of variance explained by first $k$ components:
$$\text{Explained Variance Ratio}_k = \frac{\sum_{i=1}^{k}\lambda_i}{\sum_{i=1}^{d}\lambda_i}$$

In [ ]:
# PCA projection
def pca_eig(X, n_components=None):
    """PCA using eigendecomposition."""
    X_centered = X - X.mean(axis=0)
    S = (X_centered.T @ X_centered) / (len(X) - 1)
    
    eigenvalues, eigenvectors = np.linalg.eigh(S)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    
    if n_components is None:
        n_components = len(eigenvalues)
    
    # Project
    Z = X_centered @ eigenvectors[:, :n_components]
    
    return Z, eigenvalues[:n_components], eigenvectors[:, :n_components], X_centered

# Apply to high-dimensional data
Z, eigenvals, components, X_centered = pca_eig(X, n_components=2)

print(f"Original data shape: {X.shape}")
print(f"Reduced data shape (2 components): {Z.shape}")
print(f"\nEigenvalues (first 4): {eigenvals[:4]}")
print(f"\nExplained variance ratio (first 4):")
all_eigs, _, _, _ = pca_eig(X)  # Get all eigenvalues
explained = all_eigs / all_eigs.sum()
for i, var in enumerate(explained[:4]):
    print(f"  PC{i+1}: {var:.4f}")

# Visualize projection
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Original (first 3 dims)
ax = axes[0]
ax.scatter(X[:, 0], X[:, 1], alpha=0.5, s=30)
ax.set_xlabel('Original x₀')
ax.set_ylabel('Original x₁')
ax.set_title('Original High-Dimensional Data (first 2 dims)')

# PCA projection
ax = axes[1]
scatter = ax.scatter(Z[:, 0], Z[:, 1], c=np.arctan2(Z[:, 1], Z[:, 0]), cmap='hsv', alpha=0.6, s=30)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('PCA Projection (2 components)')
ax.axhline(0, color='k', linestyle='--', alpha=0.3)
ax.axvline(0, color='k', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

## Part 4: Singular Value Decomposition (SVD)

### SVD Definition
$$\mathbf{X}_{centered} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^T$$

where:
- $\mathbf{U} \in \mathbb{R}^{n \times n}$: left singular vectors
- $\boldsymbol{\Sigma} \in \mathbb{R}^{n \times d}$: singular values (non-negative, descending)
- $\mathbf{V} \in \mathbb{R}^{d \times d}$: right singular vectors (principal components!)

### Connection to PCA
$$\mathbf{S} = \frac{1}{n-1}\mathbf{X}_{centered}^T\mathbf{X}_{centered} = \frac{1}{n-1}\mathbf{V}\boldsymbol{\Sigma}^T\boldsymbol{\Sigma}\mathbf{V}^T$$

Therefore:
- Principal components = right singular vectors ($\mathbf{V}$)
- Eigenvalues = $(\sigma_i^2) / (n-1)$

### Advantages of SVD
- **Numerically stable** (no explicit covariance computation)
- **Efficient** for tall-and-thin matrices
- Avoids issues with ill-conditioned covariance matrices

In [ ]:
# Compare eigendecomposition vs SVD
def pca_svd(X, n_components=None):
    """PCA using SVD (numerically stable)."""
    X_centered = X - X.mean(axis=0)
    
    U, S, Vt = svd(X_centered, full_matrices=False)
    
    if n_components is None:
        n_components = len(S)
    
    # Principal components are columns of V
    components = Vt[:n_components].T  # Shape: (d, n_components)
    
    # Eigenvalues from singular values
    eigenvalues = (S[:n_components] ** 2) / (len(X) - 1)
    
    # Project
    Z = X_centered @ components
    
    return Z, eigenvalues, components, X_centered

# Apply both methods
Z_eig, eigs_eig, comp_eig, _ = pca_eig(X_2d, n_components=2)
Z_svd, eigs_svd, comp_svd, _ = pca_svd(X_2d, n_components=2)

print("Eigendecomposition vs SVD Comparison:")
print(f"\nEigenvalues (Eig): {eigs_eig}")
print(f"Eigenvalues (SVD): {eigs_svd}")
print(f"Difference: {np.abs(eigs_eig - eigs_svd)}")

print(f"\nProjection difference (Frobenius norm): {np.linalg.norm(Z_eig - Z_svd):.2e}")

# Verify SVD decomposition
X_centered = X_2d - X_2d.mean(axis=0)
U, S, Vt = svd(X_centered, full_matrices=False)
X_reconstructed = U[:, :2] @ np.diag(S[:2]) @ Vt[:2, :]
reconstruction_error = np.linalg.norm(X_centered - U @ np.diag(S) @ Vt)
print(f"\nSVD reconstruction error (2/10 components): {reconstruction_error:.2e}")

## Part 5: Selecting Number of Components

### Methods
1. **Variance threshold**: Select $k$ such that explained variance ≥ threshold (e.g., 95%)
2. **Scree plot**: Visual elbow method
3. **Cross-validation**: Reconstruct task performance
4. **Information criteria**: AIC, BIC (for specific models)

In [ ]:
# Generate higher-dimensional data
n_true_dims = 5
noise_dims = 15
n = 200

# True low-rank structure
W_true = np.random.randn(n_true_dims, 10)
S_true = np.random.randn(n, n_true_dims)
X_true = S_true @ W_true + np.random.randn(n, 10) * 0.1

# Add noise
X_noisy = X_true + np.random.randn(n, 10) * 0.5

# Compute PCA
Z, eigenvals, comp, _ = pca_svd(X_noisy)

explained = eigenvals / eigenvals.sum()
cumsum = np.cumsum(explained)

# Find number of components for 95% variance
n_comp_95 = np.argmax(cumsum >= 0.95) + 1

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scree plot
ax = axes[0]
ax.bar(range(1, 11), explained, alpha=0.6, label='Individual')
ax.plot(range(1, 11), cumsum, 'ro-', linewidth=2, markersize=6, label='Cumulative')
ax.axhline(0.95, color='g', linestyle='--', linewidth=2, label='95% threshold')
ax.axvline(n_comp_95, color='orange', linestyle='--', linewidth=2, label=f'Selected K={n_comp_95}')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance')
ax.set_title('Scree Plot: Selecting Number of Components')
ax.legend()
ax.grid(True, alpha=0.3)

# Log eigenvalues (elbow method)
ax = axes[1]
ax.semilogy(range(1, 11), eigenvals, 'b-o', linewidth=2, markersize=6)
ax.axvline(n_comp_95, color='orange', linestyle='--', linewidth=2, label=f'Selected K={n_comp_95}')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Eigenvalue (log scale)')
ax.set_title('Elbow Method')
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

print(f"Explained variance (first 5 components): {cumsum[4]:.1%}")
print(f"Components needed for 95% variance: {n_comp_95}")
print(f"Dimensionality reduction: {10} → {n_comp_95} ({n_comp_95/10*100:.0f}% of original)")

## Part 6: Numerical Stability

### Data Centering and Scaling
- **Always center**: Subtract mean
- **Consider scaling**: Divide by std if dimensions have different units/scales

### Covariance vs SVD
- **Covariance + eigendecomposition**: prone to numerical errors if covariance is ill-conditioned
- **SVD**: numerically superior; no explicit covariance needed

### Condition Number
$$\kappa(\mathbf{S}) = \frac{\lambda_{max}}{\lambda_{min}}$$

High condition number → ill-conditioned → potential numerical errors.

In [ ]:
# Demonstrate numerical stability
# Create data with varying scales
X_scaled = np.random.randn(100, 3)
X_scaled[:, 0] *= 1000  # Scale first dimension
X_scaled[:, 1] *= 0.001  # Scale second dimension

# Method 1: Covariance + Eigendecomposition
X_centered = X_scaled - X_scaled.mean(axis=0)
S = (X_centered.T @ X_centered) / (len(X_scaled) - 1)
eigs_cov, _ = np.linalg.eigh(S)

# Method 2: SVD
U, S_sing, Vt = svd(X_centered, full_matrices=False)
eigs_svd = (S_sing ** 2) / (len(X_scaled) - 1)

# Condition number
kappa = np.max(eigs_cov) / np.min(eigs_cov[eigs_cov > 0])

print(f"Condition number of covariance matrix: {kappa:.2e}")
print(f"\nEigenvalues (Covariance):")
print(eigs_cov[::-1])
print(f"\nEigenvalues (SVD):")
print(eigs_svd)
print(f"\nRelative difference: {np.abs(eigs_cov[::-1] - eigs_svd) / eigs_svd}")

# Solution: Scale data first
X_standardized = (X_scaled - X_scaled.mean(axis=0)) / X_scaled.std(axis=0)
S_std = (X_standardized.T @ X_standardized) / (len(X_standardized) - 1)
eigs_std, _ = np.linalg.eigh(S_std)
kappa_std = np.max(eigs_std) / np.min(eigs_std[eigs_std > 0])

print(f"\nAfter standardization:")
print(f"Condition number: {kappa_std:.2e}")
print(f"Eigenvalues: {eigs_std[::-1]}")

## Summary

### Key Concepts
1. **Covariance matrix** captures variance and correlations
2. **Eigendecomposition** finds directions of maximum variance
3. **Principal components** are orthogonal and ordered by variance explained
4. **SVD** is numerically superior and computationally efficient
5. **Variance explained ratio** guides component selection
6. **Scaling and centering** are essential for stability

### PCA Workflow
1. Center data (subtract mean)
2. Optionally scale (divide by std)
3. Compute SVD
4. Select k components based on variance threshold
5. Project data onto principal components

### Next Steps
- **TASK-UL12 (Practical)**: Kernel PCA, face recognition with PCA, PCA preprocessing
- **Advanced**: Incremental PCA, probabilistic PCA

In [ ]:
# Exercise 1: Implement full PCA pipeline
def my_pca(X, n_components):
    """Full PCA pipeline from scratch."""
    # Step 1: Center
    mean = X.mean(axis=0)
    X_centered = X - mean
    
    # Step 2: SVD
    U, S, Vt = svd(X_centered, full_matrices=False)
    
    # Step 3: Select components
    components = Vt[:n_components].T
    
    # Step 4: Project
    Z = X_centered @ components
    
    # Explained variance
    eigenvalues = (S ** 2) / (len(X) - 1)
    explained = eigenvalues[:n_components] / eigenvalues.sum()
    
    return Z, components, mean, explained

# Test
Z, comp, mean, expl = my_pca(X_2d, 1)
print("Exercise 1: 1-component PCA projection")
print(f"Explained variance: {expl[0]:.1%}")
print(f"Projected shape: {Z.shape}")

# Exercise 2: Reconstruct data from components
print("\nExercise 2: Reconstruction error vs number of components")
for k in [1, 2, 3, 5]:
    Z_k, comp_k, mean_k, expl_k = my_pca(X_noisy, k)
    X_reconstructed = Z_k @ comp_k.T + mean_k
    reconstruction_error = np.linalg.norm(X_noisy - X_reconstructed) / np.linalg.norm(X_noisy)
    print(f"  K={k}: Relative error = {reconstruction_error:.4f}, Explained var = {expl_k.sum():.1%}")